In [36]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt

import os
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option('display.max_columns', None)

In [59]:
isw_path = Path("data/isw/isw_data.json")

isw = pd.read_json(isw_path, convert_dates=["date"]).sort_values("date", ascending=False).reset_index(drop=True)

In [60]:
isw.head()

,date,title,url,key_takeaways
0,2026-03-10,"Russian Offensive Campaign Assessment, March 1...",https://understandingwar.org/research/russia-u...,[Ukrainian forces advanced 10 to 12 kilometers...
1,2026-03-09,"Russian Offensive Campaign Assessment, March 9...",https://understandingwar.org/research/russia-u...,[Ukrainian forces are successfully counteratta...
2,2026-03-08,"Russian Offensive Campaign Assessment, March 8...",https://understandingwar.org/research/russia-u...,[Ukraine will send an unspecified number of Uk...
3,2026-03-07,"Russian Offensive Campaign Assessment, March 7...",https://understandingwar.org/research/russia-u...,[The Russian military command likely has later...
4,2026-03-06,"Russian Offensive Campaign Assessment, March 6...",https://understandingwar.org/research/russia-u...,[Russia is reportedly sharing intelligence wit...


In [61]:
isw.shape

(2107, 4)

In [62]:
isw.info()

<class 'pandas.DataFrame'>
RangeIndex: 2107 entries, 0 to 2106
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           2107 non-null   datetime64[us]
 1   title          2107 non-null   str           
 2   url            2107 non-null   str           
 3   key_takeaways  2107 non-null   object        
dtypes: datetime64[us](1), object(1), str(2)
memory usage: 385.4+ KB


In [63]:
isw.describe()

,date
count,2107
mean,2023-03-24 11:38:28.305647
min,2008-09-13 00:00:00
25%,2022-06-11 12:00:00
50%,2023-10-10 00:00:00
75%,2025-01-18 00:00:00
max,2026-03-10 00:00:00


In [64]:
isw = isw.loc[isw.date >= dt.datetime(2022, 2, 24)]

In [65]:
isw.isna().sum()

date             0
title            0
url              0
key_takeaways    0
dtype: int64

In [66]:
isw.shape

(1723, 4)

In [67]:
isw

,date,title,url,key_takeaways
0,2026-03-10,"Russian Offensive Campaign Assessment, March 1...",https://understandingwar.org/research/russia-u...,[Ukrainian forces advanced 10 to 12 kilometers...
1,2026-03-09,"Russian Offensive Campaign Assessment, March 9...",https://understandingwar.org/research/russia-u...,[Ukrainian forces are successfully counteratta...
2,2026-03-08,"Russian Offensive Campaign Assessment, March 8...",https://understandingwar.org/research/russia-u...,[Ukraine will send an unspecified number of Uk...
3,2026-03-07,"Russian Offensive Campaign Assessment, March 7...",https://understandingwar.org/research/russia-u...,[The Russian military command likely has later...
4,2026-03-06,"Russian Offensive Campaign Assessment, March 6...",https://understandingwar.org/research/russia-u...,[Russia is reportedly sharing intelligence wit...
...,...,...,...,...
1718,2022-02-26,Ukraine Conflict Update 8,https://understandingwar.org/research/russia-u...,[Russian forces entered major Ukrainian cities...
1719,2022-02-25,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,[Russian forces entered the outskirts of Kyiv ...
1720,2022-02-25,Ukraine Conflict Update 7,https://understandingwar.org/research/russia-u...,[Ukrainian forces are successfully slowing Rus...
1721,2022-02-24,Russia-Ukraine Warning Update: Initial Russian...,https://understandingwar.org/research/russia-u...,[Ukrainian forces are successfully slowing Rus...


In [68]:
for i in isw.index:
    isw.loc[i, "key_takeaways"] = " ".join(isw.loc[i, "key_takeaways"])

In [69]:
isw

,date,title,url,key_takeaways
0,2026-03-10,"Russian Offensive Campaign Assessment, March 1...",https://understandingwar.org/research/russia-u...,Ukrainian forces advanced 10 to 12 kilometers ...
1,2026-03-09,"Russian Offensive Campaign Assessment, March 9...",https://understandingwar.org/research/russia-u...,Ukrainian forces are successfully counterattac...
2,2026-03-08,"Russian Offensive Campaign Assessment, March 8...",https://understandingwar.org/research/russia-u...,Ukraine will send an unspecified number of Ukr...
3,2026-03-07,"Russian Offensive Campaign Assessment, March 7...",https://understandingwar.org/research/russia-u...,The Russian military command likely has latera...
4,2026-03-06,"Russian Offensive Campaign Assessment, March 6...",https://understandingwar.org/research/russia-u...,Russia is reportedly sharing intelligence with...
...,...,...,...,...
1718,2022-02-26,Ukraine Conflict Update 8,https://understandingwar.org/research/russia-u...,Russian forces entered major Ukrainian cities—...
1719,2022-02-25,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,Russian forces entered the outskirts of Kyiv o...
1720,2022-02-25,Ukraine Conflict Update 7,https://understandingwar.org/research/russia-u...,Ukrainian forces are successfully slowing Russ...
1721,2022-02-24,Russia-Ukraine Warning Update: Initial Russian...,https://understandingwar.org/research/russia-u...,Ukrainian forces are successfully slowing Russ...


In [70]:
isw.sample(5)

,date,title,url,key_takeaways
348,2025-06-04,"Russian Offensive Campaign Assessment, June 4,...",https://understandingwar.org/research/russia-u...,The Kremlin is fixating on recent train derail...
1468,2022-09-28,"Russian Offensive Campaign Assessment, Septemb...",https://understandingwar.org/research/russia-u...,Russian military leadership has likely failed ...
965,2023-12-24,"Russian Offensive Campaign Assessment, Decembe...",https://understandingwar.org/research/russia-u...,European Union (EU) Foreign Affairs High Repre...
1182,2023-06-27,"Russian Offensive Campaign Assessment, June 26...",https://understandingwar.org/research/russia-u...,Russian President Vladimir Putin gave a speech...
183,2025-10-04,"Russian Offensive Campaign Assessment, October...",https://understandingwar.org/research/russia-u...,German officials reported more unidentified dr...


In [71]:
isw.loc[isw.key_takeaways == "no key takeways", "key_takeaways"] = None

In [72]:
isw.isna().sum()

date               0
title              0
url                0
key_takeaways    213
dtype: int64

In [74]:
isw.loc[isw.key_takeaways.isna()]

,date,title,url,key_takeaways
17,2026-02-24,Putin’s Internet Crackdown Is Rooted in Weakne...,https://understandingwar.org/research/cognitiv...,None
18,2026-02-24,The West Shouldn’t Play Russia’s Game with Ukr...,https://understandingwar.org/research/russia-u...,None
20,2026-02-23,Russia’s Quest to Intensify The Theater-Wide B...,https://understandingwar.org/research/russia-u...,None
66,2026-01-15,Russia’s Non-Response to US Actions in Venezue...,https://understandingwar.org/research/adversar...,None
130,2025-11-16,Seizing the Initiative against Russia: Putting...,https://understandingwar.org/research/russia-u...,None
...,...,...,...,...
1698,2022-03-06,The World After the War in Ukraine: Who or Wha...,https://understandingwar.org/research/russia-u...,None
1700,2022-03-05,"Explainer on Russian Conscription, Reserve, an...",https://understandingwar.org/research/russia-u...,None
1711,2022-03-01,Ukraine Conflict Update 11,https://understandingwar.org/research/russia-u...,None
1713,2022-02-28,Ukraine Conflict Update 10,https://understandingwar.org/research/russia-u...,None
